In [8]:
import os
import json
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from nilearn.glm.first_level import FirstLevelModel
from nilearn.plotting import plot_design_matrix, plot_stat_map


In [ ]:
# Subject
subject = "sub-002"

# base_dir = os.path.expanduser("~/neu502b/langloc_glm_work")
data_dir = "/usr/people/zt4569/neu502b/neu502b_fmri/final_proj_localizer"
events_dir = "/usr/people/zt4569/neu502b/neu502b_fmri/final_proj_localizer"
output_dir = "/usr/people/zt4569/neu502b/neu502b_fmri/final_proj_localizer"

os.makedirs(output_dir, exist_ok=True)

bold_file = os.path.join(
    data_dir,
    f"{subject}_ses-01_task-langXlocal_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
)

# bold_json = os.path.join(
#     data_dir,
#     f"{subject}_ses-01_task-langXlocal_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.json"
# )

mask_file = os.path.join(
    data_dir,
    f"{subject}_ses-01_task-langXlocal_run-1_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz"
)

confounds_file = os.path.join(
    data_dir,
    f"{subject}_ses-01_task-langXlocal_run-1_desc-confounds_timeseries.tsv"
)

events_file = os.path.join(
    events_dir,
    f"langloc_events.tsv"
)


In [10]:
# Load BOLD + TR
bold_img = nib.load(bold_file)
n_scans = bold_img.shape[-1]

# tr = None
# if os.path.exists(bold_json):
#     with open(bold_json, "r") as f:
#         meta = json.load(f)
#     tr = meta.get("RepetitionTime", None)

# if tr is None:
#     tr = bold_img.header.get_zooms()[-1]

print(f"Subject: {subject}")
print(f"BOLD shape: {bold_img.shape}")
print(f"n_scans: {n_scans}")
# print(f"TR: {tr}")

Subject: sub-002
BOLD shape: (78, 93, 78, 180)
n_scans: 180


In [11]:
# Load events
events_df = pd.read_csv(events_file, sep="\t")

required_cols = {"onset", "duration", "trial_type"}
missing = required_cols - set(events_df.columns)
if missing:
    raise ValueError(f"Events file missing columns: {missing}")

print("\nEvents preview:")
print(events_df.head())
print("\nTrial counts:")
print(events_df["trial_type"].value_counts())


Events preview:
   onset trial_type  duration
0     14          S         4
1     20          S         4
2     26          S         4
3     32          N         4
4     38          N         4

Trial counts:
trial_type
S    24
N    24
Name: count, dtype: int64


In [12]:
# Load confounds
confounds_df = pd.read_csv(confounds_file, sep="\t")

motion_cols = [
    "trans_x", "trans_y", "trans_z",
    "rot_x", "rot_y", "rot_z"
]

acompcor_cols = [c for c in confounds_df.columns if c.startswith("a_comp_cor")][:5]

selected_confounds = []
for c in motion_cols:
    if c in confounds_df.columns:
        selected_confounds.append(c)

for c in acompcor_cols:
    if c in confounds_df.columns:
        selected_confounds.append(c)

if "framewise_displacement" in confounds_df.columns:
    selected_confounds.append("framewise_displacement")

if not selected_confounds:
    raise ValueError("No confound columns were found.")

confounds_use = confounds_df[selected_confounds].copy()
confounds_use = confounds_use.fillna(0)

if len(confounds_use) != n_scans:
    raise ValueError(
        f"Confounds rows ({len(confounds_use)}) do not match n_scans ({n_scans})."
    )

print("\nConfounds used:")
print(confounds_use.columns.tolist())
print(confounds_use.shape)



Confounds used:
['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03', 'a_comp_cor_04', 'framewise_displacement']
(180, 12)


In [13]:
# Fit GLM
glm = FirstLevelModel(
    t_r=2,
    mask_img=mask_file,
    noise_model="ar1",
    standardize=False,
    hrf_model="glover",
    drift_model="cosine",
    high_pass=1/128,
    smoothing_fwhm=5.0
)

glm = glm.fit(
    run_imgs=bold_file,
    events=events_df,
    confounds=confounds_use
)


In [14]:
# Extract + save design matrix
design_matrix = glm.design_matrices_[0]

print("\nDesign matrix columns:")
print(design_matrix.columns.tolist())

plt.figure(figsize=(14, 6))
plot_design_matrix(design_matrix)
plt.tight_layout()
design_png = os.path.join(output_dir, f"{subject}_design_matrix.png")
plt.savefig(design_png, dpi=200, bbox_inches="tight")
plt.close()

print(f"\nSaved design matrix PNG to:\n{design_png}")



Design matrix columns:
['N', 'S', 'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03', 'a_comp_cor_04', 'framewise_displacement', 'drift_1', 'drift_2', 'drift_3', 'drift_4', 'drift_5', 'constant']


/tmp/ipykernel_3617203/500136149.py:9: UserWarning: The figure layout has changed to tight
  plt.tight_layout()



Saved design matrix PNG to:
/usr/people/zt4569/neu502b/final_proj_localizer/sub-002_design_matrix.png


<Figure size 1400x600 with 0 Axes>

In [15]:
# Compute contrasts
# Main localizer contrast
contrast_expr = "S - N"

z_map = glm.compute_contrast(contrast_expr, output_type="z_score")
effect_map = glm.compute_contrast(contrast_expr, output_type="effect_size")

zmap_file = os.path.join(output_dir, f"{subject}_S_minus_N_zmap.nii.gz")
effect_file = os.path.join(output_dir, f"{subject}_S_minus_N_effect.nii.gz")

z_map.to_filename(zmap_file)
effect_map.to_filename(effect_file)

print(f"Saved z-map to:\n{zmap_file}")
print(f"Saved effect map to:\n{effect_file}")


Saved z-map to:
/usr/people/zt4569/neu502b/final_proj_localizer/sub-002_S_minus_N_zmap.nii.gz
Saved effect map to:
/usr/people/zt4569/neu502b/final_proj_localizer/sub-002_S_minus_N_effect.nii.gz


In [16]:
# Save visualization
display = plot_stat_map(
    z_map,
    threshold=3.1,
    display_mode="z",
    cut_coords=6,
    black_bg=False,
    title=f"{subject}: S > N"
)

stat_png = os.path.join(output_dir, f"{subject}_S_minus_N_zmap.png")
display.savefig(stat_png, dpi=200)
display.close()

print(f"Saved stat map PNG to:\n{stat_png}")
print("\nDone.")

Saved stat map PNG to:
/usr/people/zt4569/neu502b/final_proj_localizer/sub-002_S_minus_N_zmap.png

Done.
